# Step 18 — Harmonize signature annotations with reviewed tumor-cluster annotations

This notebook combines two complementary annotation strategies:

```text
Existing signature-based broad annotation
    obs["prelim_cell_type_primary_tumor_expanded"]

Reviewed cancer-specific Leiden 0.2 annotations
    melanoma / NSCLC / colon_cancer cluster workbenches
```

It creates two all-cell annotation columns:

```python
obs["Specific_celltype_annotation"]
obs["Broader_celltype_annotation"]
```

## Mixed-annotation logic

For cells included in the cancer-specific `Tumor/epithelial + Other/unresolved`
Leiden analysis:

```text
Specific_celltype_annotation
    ← workbench column "Specific annotation"

Broader_celltype_annotation
    ← workbench column "Broader annotation"
```

For every other cell:

```text
both new columns
    ← existing prelim_cell_type_primary_tumor_expanded
```

Thus, resolved immune, stromal, endothelial, and other signature-based calls
are preserved, while reviewed Leiden-0.2 annotations refine the difficult
tumor/unresolved compartment.

Cells that were not included in Step 16—such as QC-failing tumor/unresolved
cells—retain their signature-based label rather than receiving an unsupported
cluster annotation.

## Main outputs

```text
all_cells_mixed_annotation_handoff.parquet
all_cells_mixed_annotation_handoff.csv.gz

all_cells_annotation_update_minimal.parquet
all_cells_annotation_update_minimal.csv.gz

reviewed_tumor_cluster_annotation_map.csv
celltype_color_palette.csv
celltype_color_palette.json

one Specific and one Broader spatial plot per sample
one two-panel comparison plot per sample

Specific and Broader stacked-composition figures:
    all cells
    Screen versus C2D15
    patient
    sample

count and fraction tables for every grouping
```

The notebook reads only `.obs` and `obsm["spatial"]` from the large merged
handoff Zarr; it does not materialize the dense corrected-expression matrix.


In [1]:
# ---------------------------------------------------------------------
# Imports and configuration
# ---------------------------------------------------------------------
from __future__ import annotations

import gc
import json
import math
import re
import traceback
import warnings
from collections import OrderedDict
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd

from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from matplotlib.ticker import PercentFormatter
from IPython.display import display

plt.ioff()

PROJECT_ROOT = Path(
    "/host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057"
)
PIPELINE_ROOT = (
    PROJECT_ROOT
    / "tmp"
    / "proseg_resolvi_immune_enrichment_v1"
)

STEP16_ROOT = (
    PIPELINE_ROOT
    / "16_tumor_unresolved_leiden_multiresolution_selectionfix_v2"
)
STEP17_ROOT = (
    PIPELINE_ROOT
    / "17_tumor_unresolved_resolution0p2_deep_review"
)
OUTPUT_ROOT = (
    PIPELINE_ROOT
    / "18_harmonized_signature_and_tumor_cluster_annotations"
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

MERGED_HANDOFF_FILENAME = (
    "VisiumHD_12samples_ResolVI_corrected_preliminary_annotation.zarr"
)
BASELINE_ANNOTATION_COLUMN = (
    "prelim_cell_type_primary_tumor_expanded"
)
CLUSTER_KEY = "tumor_leiden_res_0_2"

SPECIFIC_WORKBENCH_COLUMN = "Specific annotation"
BROADER_WORKBENCH_COLUMN = "Broader annotation"

# Standardized output names. The broader column intentionally contains no
# embedded space before "_annotation".
SPECIFIC_OUTPUT_COLUMN = "Specific_celltype_annotation"
BROADER_OUTPUT_COLUMN = "Broader_celltype_annotation"

CANCER_TYPES = [
    "melanoma",
    "NSCLC",
    "colon_cancer",
]

# Optional exact path overrides when a manually edited workbench was saved
# under a different filename.
WORKBENCH_PATH_OVERRIDES = {
    # "melanoma": Path("/absolute/path/to/edited.csv"),
    # "NSCLC": Path("/absolute/path/to/edited.csv"),
    # "colon_cancer": Path("/absolute/path/to/edited.csv"),
}

# Optional exact terminology harmonization applied to both reviewed columns.
# Keys must match the reviewed annotation after surrounding whitespace is
# removed.
LABEL_REPLACEMENTS = {
    # "Macrophage": "Monocyte/macrophage",
}

ALLOW_INCOMPLETE_WORKBENCH = False

# Spatial plots.
SPATIAL_POINT_SIZE = 5.0
SPATIAL_ALPHA = 0.82
SPATIAL_MAX_FIGURE_INCHES = 13.0
SPATIAL_MIN_FIGURE_INCHES = 6.0
PLOT_DPI = 350

# Composition plots.
MAKE_FRACTION_STACKED_BARS = True
MAKE_COUNT_STACKED_BARS = True
STACKED_BAR_LEGEND_COLUMNS = 5

# A single palette is shared across both annotation columns and all figures.
CUSTOM_COLOR_OVERRIDES = {
    # "Tumor": "#D73027",
}

OVERWRITE = True
PIPELINE_VERSION = (
    "2026-08-21-harmonized-signature-tumor-cluster-annotations-v1"
)

print("Step 16 root:", STEP16_ROOT)
print("Step 17 root:", STEP17_ROOT)
print("Output root:", OUTPUT_ROOT)
print("Specific output column:", SPECIFIC_OUTPUT_COLUMN)
print("Broader output column:", BROADER_OUTPUT_COLUMN)


Step 16 root: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/16_tumor_unresolved_leiden_multiresolution_selectionfix_v2
Step 17 root: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/17_tumor_unresolved_resolution0p2_deep_review
Output root: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/18_harmonized_signature_and_tumor_cluster_annotations
Specific output column: Specific_celltype_annotation
Broader output column: Broader_celltype_annotation


In [2]:
# ---------------------------------------------------------------------
# Sample metadata and display order
# ---------------------------------------------------------------------
SAMPLE_INFO = OrderedDict(
    {
        # Melanoma
        "Screen_16_22": {
            "patient": "patient_16_22",
            "cancer_type": "melanoma",
            "biopsy_stage": "Screen",
        },
        "C2D15_16_22": {
            "patient": "patient_16_22",
            "cancer_type": "melanoma",
            "biopsy_stage": "C2D15",
        },
        "Screen_18_23": {
            "patient": "patient_18_23",
            "cancer_type": "melanoma",
            "biopsy_stage": "Screen",
        },
        "C2D15_18_23": {
            "patient": "patient_18_23",
            "cancer_type": "melanoma",
            "biopsy_stage": "C2D15",
        },
        "Screen_30_16": {
            "patient": "patient_30_16",
            "cancer_type": "melanoma",
            "biopsy_stage": "Screen",
        },
        "C2D15_30_16": {
            "patient": "patient_30_16",
            "cancer_type": "melanoma",
            "biopsy_stage": "C2D15",
        },

        # NSCLC
        "Screen_17_26": {
            "patient": "patient_17_26",
            "cancer_type": "NSCLC",
            "biopsy_stage": "Screen",
        },
        "C2D15_17_26": {
            "patient": "patient_17_26",
            "cancer_type": "NSCLC",
            "biopsy_stage": "C2D15",
        },
        "Screen_39_21": {
            "patient": "patient_39_21",
            "cancer_type": "NSCLC",
            "biopsy_stage": "Screen",
        },
        "C2D15_39_21": {
            "patient": "patient_39_21",
            "cancer_type": "NSCLC",
            "biopsy_stage": "C2D15",
        },

        # MSS-CRC
        "Screen_23_25": {
            "patient": "patient_23_25",
            "cancer_type": "colon_cancer",
            "biopsy_stage": "Screen",
        },
        "C2D15_23_25": {
            "patient": "patient_23_25",
            "cancer_type": "colon_cancer",
            "biopsy_stage": "C2D15",
        },
    }
)

SAMPLE_ORDER = list(SAMPLE_INFO)
PATIENT_ORDER = list(
    OrderedDict(
        (
            metadata["patient"],
            None,
        )
        for metadata in SAMPLE_INFO.values()
    )
)
STAGE_ORDER = ["Screen", "C2D15"]

PREFERRED_CELLTYPE_ORDER = [
    "Treg",
    "CD4+ T",
    "CD8+ T",
    "Other T",
    "NK",
    "B/plasma",
    "Monocyte/macrophage",
    "Fibroblast/stromal",
    "Endothelial",
    "Keratinocyte",
    "Tumor/epithelial",
    "Tumor",
    "Other/unresolved",
]

KNOWN_CELLTYPE_COLORS = {
    "Treg": "#7B2CBF",
    "CD4+ T": "#4D96FF",
    "CD8+ T": "#1F5AA6",
    "Other T": "#8ECAE6",
    "NK": "#00A6A6",
    "B/plasma": "#F4A261",
    "Monocyte/macrophage": "#8C564B",
    "Fibroblast/stromal": "#E9C46A",
    "Endothelial": "#2A9D8F",
    "Keratinocyte": "#F28482",
    "Tumor/epithelial": "#E63946",
    "Tumor": "#D73027",
    "Other/unresolved": "#9E9E9E",
}

print("Sample order:", SAMPLE_ORDER)
print("Patient order:", PATIENT_ORDER)


Sample order: ['Screen_16_22', 'C2D15_16_22', 'Screen_18_23', 'C2D15_18_23', 'Screen_30_16', 'C2D15_30_16', 'Screen_17_26', 'C2D15_17_26', 'Screen_39_21', 'C2D15_39_21', 'Screen_23_25', 'C2D15_23_25']
Patient order: ['patient_16_22', 'patient_18_23', 'patient_30_16', 'patient_17_26', 'patient_39_21', 'patient_23_25']


In [3]:
# ---------------------------------------------------------------------
# Paths and normalization helpers
# ---------------------------------------------------------------------
def cancer_slug(cancer_type: str) -> str:
    return (
        str(cancer_type)
        .lower()
        .replace(" ", "_")
        .replace("/", "_")
    )


def canonical_column_token(value: str) -> str:
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(value).strip().lower(),
    )


def canonical_cluster_id(value) -> str:
    if pd.isna(value):
        return ""

    text = str(value).strip()
    text = re.sub(
        r"^\s*cluster\s*",
        "",
        text,
        flags=re.IGNORECASE,
    )

    try:
        number = float(text)
        if np.isfinite(number) and number.is_integer():
            return str(int(number))
    except Exception:
        pass

    return text


def clean_annotation_value(value) -> str:
    if pd.isna(value):
        return ""

    text = re.sub(
        r"\s+",
        " ",
        str(value).strip(),
    )

    if text.lower() in {
        "",
        "nan",
        "none",
        "null",
        "na",
        "n/a",
    }:
        return ""

    return LABEL_REPLACEMENTS.get(text, text)


def find_column(
    frame: pd.DataFrame,
    preferred: str,
    aliases: list[str],
) -> str:
    if preferred in frame.columns:
        return preferred

    token_to_columns: dict[str, list[str]] = {}
    for column in frame.columns:
        token_to_columns.setdefault(
            canonical_column_token(column),
            [],
        ).append(str(column))

    candidates = [
        preferred,
        *aliases,
    ]

    matches = []
    for candidate in candidates:
        matches.extend(
            token_to_columns.get(
                canonical_column_token(candidate),
                [],
            )
        )

    matches = list(dict.fromkeys(matches))

    if len(matches) == 1:
        return matches[0]

    raise KeyError(
        f"Could not resolve column {preferred!r}. "
        f"Aliases={aliases}. Available columns={list(frame.columns)}. "
        f"Matches={matches}"
    )


def discover_merged_handoff() -> Path:
    expected = (
        PIPELINE_ROOT
        / "15_merged_annotation_handoff"
        / MERGED_HANDOFF_FILENAME
    )
    if expected.exists():
        return expected

    candidates = sorted(
        PIPELINE_ROOT.rglob(
            MERGED_HANDOFF_FILENAME
        )
    )
    if len(candidates) == 1:
        return candidates[0]
    if len(candidates) == 0:
        raise FileNotFoundError(
            f"Could not locate {MERGED_HANDOFF_FILENAME} "
            f"under {PIPELINE_ROOT}"
        )
    raise RuntimeError(
        "Multiple merged handoff candidates were found. "
        "Set an explicit path in discover_merged_handoff():\n"
        + "\n".join(str(path) for path in candidates)
    )


def workbench_path(cancer_type: str) -> Path:
    if cancer_type in WORKBENCH_PATH_OVERRIDES:
        path = Path(
            WORKBENCH_PATH_OVERRIDES[cancer_type]
        )
        if not path.exists():
            raise FileNotFoundError(path)
        return path

    slug = cancer_slug(cancer_type)
    exact = (
        STEP17_ROOT
        / slug
        / "tables"
        / f"{slug}_resolution0p2_cluster_annotation_workbench.csv"
    )
    if exact.exists():
        return exact

    candidates = sorted(
        (
            STEP17_ROOT
            / slug
        ).rglob(
            "*resolution0p2*annotation*workbench*.csv"
        )
    )
    if len(candidates) == 1:
        return candidates[0]
    if len(candidates) == 0:
        raise FileNotFoundError(
            f"No resolution-0.2 annotation workbench found for "
            f"{cancer_type} under {STEP17_ROOT / slug}"
        )
    raise RuntimeError(
        f"Multiple workbench candidates found for {cancer_type}:\n"
        + "\n".join(str(path) for path in candidates)
    )


def assignment_path(cancer_type: str) -> Path:
    slug = cancer_slug(cancer_type)
    exact = (
        STEP16_ROOT
        / slug
        / "tables"
        / f"{slug}_tumor_unresolved_cluster_assignments.parquet"
    )
    if exact.exists():
        return exact

    candidates = sorted(
        (
            STEP16_ROOT
            / slug
        ).rglob(
            "*tumor_unresolved*cluster_assignments.parquet"
        )
    )
    if len(candidates) == 1:
        return candidates[0]
    if len(candidates) == 0:
        raise FileNotFoundError(
            f"No Step 16 cluster-assignment Parquet found for "
            f"{cancer_type} under {STEP16_ROOT / slug}"
        )
    raise RuntimeError(
        f"Multiple assignment candidates found for {cancer_type}:\n"
        + "\n".join(str(path) for path in candidates)
    )


MERGED_HANDOFF_PATH = discover_merged_handoff()

print("Merged handoff:", MERGED_HANDOFF_PATH)
for cancer_type in CANCER_TYPES:
    print(
        f"{cancer_type:14s}",
        "workbench=",
        workbench_path(cancer_type),
    )
    print(
        f"{'':14s}",
        "assignments=",
        assignment_path(cancer_type),
    )


Merged handoff: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/15_merged_annotation_handoff/VisiumHD_12samples_ResolVI_corrected_preliminary_annotation.zarr
melanoma       workbench= /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/17_tumor_unresolved_resolution0p2_deep_review/melanoma/tables/melanoma_resolution0p2_cluster_annotation_workbench.csv
               assignments= /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/16_tumor_unresolved_leiden_multiresolution_selectionfix_v2/melanoma/tables/melanoma_tumor_unresolved_cluster_assignments.parquet
NSCLC          workbench= /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/17_tumor_unresolved_resolution0p2_deep_review/nsclc/tables/nsclc_resolution0p2_cluster_annotation_workbench.csv
               assignments= /host_root/nethome/

## Load only metadata and spatial coordinates from the merged handoff

The merged handoff contains a dense corrected expression matrix, but this
annotation/plotting notebook does not need it.

The cell metadata and sample-local spatial coordinates are loaded lazily:

```python
obs
obsm["spatial"]
```

The merged index is retained as the authoritative all-sample cell ID.


In [4]:
# ---------------------------------------------------------------------
# Load merged obs and sample-local spatial coordinates without loading X
# ---------------------------------------------------------------------
def materialize_frame(value) -> pd.DataFrame:
    if hasattr(value, "to_memory"):
        value = value.to_memory()
    if hasattr(value, "compute"):
        value = value.compute()
    return pd.DataFrame(value).copy()


def materialize_array(value) -> np.ndarray:
    if hasattr(value, "to_memory"):
        value = value.to_memory()
    if hasattr(value, "compute"):
        value = value.compute()
    return np.asarray(value)


if not hasattr(ad.experimental, "read_lazy"):
    raise RuntimeError(
        "This notebook requires anndata.experimental.read_lazy() "
        "to avoid materializing the dense corrected-expression matrix."
    )

merged_lazy = ad.experimental.read_lazy(
    str(MERGED_HANDOFF_PATH)
)

merged_obs = materialize_frame(
    merged_lazy.obs
)
merged_obs.index = merged_obs.index.astype(str)
merged_obs.index.name = "cell_id"

if BASELINE_ANNOTATION_COLUMN not in merged_obs:
    raise KeyError(
        f"Merged handoff lacks {BASELINE_ANNOTATION_COLUMN!r}."
    )

for required in [
    "sample",
    "patient",
    "cancer_type",
    "biopsy_stage",
]:
    if required not in merged_obs:
        raise KeyError(
            f"Merged handoff lacks obs[{required!r}]."
        )

if "spatial" not in merged_lazy.obsm:
    raise KeyError(
        "Merged handoff lacks obsm['spatial']."
    )

merged_spatial = materialize_array(
    merged_lazy.obsm["spatial"]
).astype(np.float32, copy=False)

if merged_spatial.shape != (
    len(merged_obs),
    2,
):
    raise ValueError(
        f"Unexpected spatial shape {merged_spatial.shape}; "
        f"expected {(len(merged_obs), 2)}."
    )

# Normalize metadata strings without changing the scientific labels.
for column in [
    "sample",
    "patient",
    "cancer_type",
    "biopsy_stage",
    BASELINE_ANNOTATION_COLUMN,
]:
    merged_obs[column] = (
        merged_obs[column]
        .astype("string")
        .fillna("missing")
        .astype(str)
    )

unknown_samples = sorted(
    set(merged_obs["sample"])
    .difference(SAMPLE_INFO)
)
if unknown_samples:
    raise RuntimeError(
        f"Merged handoff contains unexpected samples: {unknown_samples}"
    )

print("Merged cells:", f"{len(merged_obs):,}")
print("Spatial shape:", merged_spatial.shape)
print("Baseline annotations:")
display(
    merged_obs[
        BASELINE_ANNOTATION_COLUMN
    ]
    .value_counts()
    .rename("n_cells")
    .to_frame()
)


/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/lib/python3.13/site-packages/anndata/_core/xarray.py:32: UserWarning: Did not read zarr as consolidated. Consider consolidating your metadata.
  return func(*args, **kwargs)


Merged cells: 862,427
Spatial shape: (862427, 2)
Baseline annotations:


,n_cells
prelim_cell_type_primary_tumor_expanded,
Other/unresolved,575710
Tumor/epithelial,112818
Monocyte/macrophage,58327
Fibroblast/stromal,49859
Endothelial,23180
B/plasma,22907
Other T,10491
NK,4429
CD4+ T,2105


In [5]:
# ---------------------------------------------------------------------
# Read and validate the three reviewed workbenches
# ---------------------------------------------------------------------
workbench_rows = []
workbench_audit_rows = []

for cancer_type in CANCER_TYPES:
    path = workbench_path(cancer_type)
    frame = pd.read_csv(path)

    cluster_column = find_column(
        frame,
        "cluster",
        [
            "Cluster",
            "Leiden cluster",
            CLUSTER_KEY,
        ],
    )
    specific_column = find_column(
        frame,
        SPECIFIC_WORKBENCH_COLUMN,
        [
            "Specific_annotation",
            "Specific celltype annotation",
            "Specific_celltype_annotation",
            "Reviewed specific annotation",
        ],
    )
    broader_column = find_column(
        frame,
        BROADER_WORKBENCH_COLUMN,
        [
            "Broader_annotation",
            "Broader celltype annotation",
            "Broader_celltype_annotation",
            "Reviewed broader annotation",
        ],
    )

    mapping = pd.DataFrame(
        {
            "cancer_type": cancer_type,
            "cluster": frame[
                cluster_column
            ].map(canonical_cluster_id),
            "Specific_annotation_reviewed": frame[
                specific_column
            ].map(clean_annotation_value),
            "Broader_annotation_reviewed": frame[
                broader_column
            ].map(clean_annotation_value),
        }
    )

    mapping = mapping.loc[
        mapping["cluster"].ne("")
    ].copy()

    if mapping["cluster"].duplicated().any():
        duplicates = mapping.loc[
            mapping["cluster"].duplicated(
                keep=False
            )
        ]
        raise ValueError(
            f"{cancer_type}: duplicate cluster rows in {path}:\n"
            + duplicates.to_string(index=False)
        )

    incomplete = mapping[
        mapping[
            [
                "Specific_annotation_reviewed",
                "Broader_annotation_reviewed",
            ]
        ]
        .eq("")
        .any(axis=1)
    ]

    if len(incomplete) and not ALLOW_INCOMPLETE_WORKBENCH:
        raise ValueError(
            f"{cancer_type}: reviewed annotations are incomplete in "
            f"{path}:\n"
            + incomplete.to_string(index=False)
        )

    mapping["workbench_path"] = str(path)
    workbench_rows.append(mapping)

    workbench_audit_rows.append(
        {
            "cancer_type": cancer_type,
            "workbench": str(path),
            "cluster_column": cluster_column,
            "specific_column": specific_column,
            "broader_column": broader_column,
            "n_clusters": int(len(mapping)),
            "n_missing_specific": int(
                mapping[
                    "Specific_annotation_reviewed"
                ].eq("").sum()
            ),
            "n_missing_broader": int(
                mapping[
                    "Broader_annotation_reviewed"
                ].eq("").sum()
            ),
        }
    )

reviewed_cluster_map = pd.concat(
    workbench_rows,
    ignore_index=True,
)
workbench_audit = pd.DataFrame(
    workbench_audit_rows
)

display(workbench_audit)
display(reviewed_cluster_map)

reviewed_cluster_map.to_csv(
    OUTPUT_ROOT
    / "reviewed_tumor_cluster_annotation_map.csv",
    index=False,
)
workbench_audit.to_csv(
    OUTPUT_ROOT
    / "reviewed_workbench_input_audit.csv",
    index=False,
)


,cancer_type,workbench,cluster_column,specific_column,broader_column,n_clusters,n_missing_specific,n_missing_broader
0,melanoma,/host_root/nethome/reny28/Projects/Visium_proj...,cluster,Specific annotation,Broader annotation,19,0,0
1,NSCLC,/host_root/nethome/reny28/Projects/Visium_proj...,cluster,Specific annotation,Broader annotation,10,0,0
2,colon_cancer,/host_root/nethome/reny28/Projects/Visium_proj...,cluster,Specific annotation,Broader annotation,10,0,0


,cancer_type,cluster,Specific_annotation_reviewed,Broader_annotation_reviewed,workbench_path
0,melanoma,0,Tumor/differentiated,Tumor,/host_root/nethome/reny28/Projects/Visium_proj...
1,melanoma,1,Endothelial,Endothelial,/host_root/nethome/reny28/Projects/Visium_proj...
2,melanoma,2,Endothelial,Endothelial,/host_root/nethome/reny28/Projects/Visium_proj...
3,melanoma,3,Tumor/proliferation,Tumor,/host_root/nethome/reny28/Projects/Visium_proj...
4,melanoma,4,Fibroblast/stromal,Fibroblast/stromal,/host_root/nethome/reny28/Projects/Visium_proj...
5,melanoma,5,Keratinocyte,Keratinocyte,/host_root/nethome/reny28/Projects/Visium_proj...
6,melanoma,6,Tumor/proliferation,Tumor,/host_root/nethome/reny28/Projects/Visium_proj...
7,melanoma,7,Tumor/stressed,Tumor,/host_root/nethome/reny28/Projects/Visium_proj...
8,melanoma,8,Tumor/differentiated,Tumor,/host_root/nethome/reny28/Projects/Visium_proj...
9,melanoma,9,Tumor/AP-1,Tumor,/host_root/nethome/reny28/Projects/Visium_proj...


In [6]:
# ---------------------------------------------------------------------
# Load Step 16 cell-to-cluster assignments and attach reviewed labels
# ---------------------------------------------------------------------
assignment_frames = []
assignment_audit_rows = []

for cancer_type in CANCER_TYPES:
    path = assignment_path(cancer_type)
    assignments = pd.read_parquet(path)

    if "cell_id" in assignments.columns:
        assignments = assignments.set_index(
            "cell_id",
            drop=True,
        )

    assignments.index = (
        assignments.index
        .astype(str)
    )
    assignments.index.name = "cell_id"

    if CLUSTER_KEY not in assignments:
        raise KeyError(
            f"{cancer_type}: {path} lacks {CLUSTER_KEY!r}."
        )

    # Prefer the merged-style index already written by Step 16. When older
    # outputs use a local ID, reconstruct <sample>::<source_cell_id>.
    direct_overlap = float(
        assignments.index.isin(
            merged_obs.index
        ).mean()
    )

    id_method = "assignment_index"
    if direct_overlap < 0.999:
        if {
            "sample",
            "source_cell_id",
        }.issubset(assignments.columns):
            rebuilt = (
                assignments["sample"].astype(str)
                + "::"
                + assignments[
                    "source_cell_id"
                ].astype(str)
            )
            rebuilt_overlap = float(
                rebuilt.isin(
                    merged_obs.index
                ).mean()
            )
            if rebuilt_overlap > direct_overlap:
                assignments.index = pd.Index(
                    rebuilt,
                    name="cell_id",
                )
                direct_overlap = rebuilt_overlap
                id_method = (
                    "sample_double_colon_source_cell_id"
                )

    if direct_overlap < 1.0:
        examples = assignments.index[
            ~assignments.index.isin(
                merged_obs.index
            )
        ][:10].tolist()
        raise RuntimeError(
            f"{cancer_type}: only {direct_overlap:.3%} of Step 16 "
            f"assignment IDs occur in the merged handoff. "
            f"Missing examples={examples}"
        )

    assignments = assignments.copy()

    # Preserve the authoritative merged-cell ID across the column-based merge.
    # pandas.DataFrame.merge() otherwise replaces the index with a RangeIndex.
    assignments["cell_id"] = (
        assignments.index.astype(str)
    )
    assignments["cancer_type_cluster_object"] = (
        cancer_type
    )
    assignments["cluster"] = assignments[
        CLUSTER_KEY
    ].map(canonical_cluster_id)

    mapping = reviewed_cluster_map[
        reviewed_cluster_map[
            "cancer_type"
        ].eq(cancer_type)
    ][
        [
            "cluster",
            "Specific_annotation_reviewed",
            "Broader_annotation_reviewed",
        ]
    ]

    assignments = assignments.merge(
        mapping,
        on="cluster",
        how="left",
        validate="many_to_one",
        sort=False,
    )
    assignments = assignments.set_index(
        "cell_id",
        drop=True,
    )
    assignments.index = pd.Index(
        assignments.index.astype(str),
        name="cell_id",
    )

    if not assignments.index.is_unique:
        duplicates = assignments.index[
            assignments.index.duplicated(
                keep=False
            )
        ][:20].tolist()
        raise RuntimeError(
            f"{cancer_type}: duplicate cell IDs after joining the "
            f"workbench mapping: {duplicates}"
        )

    missing_review = assignments[
        [
            "Specific_annotation_reviewed",
            "Broader_annotation_reviewed",
        ]
    ].isna().any(axis=1)

    if missing_review.any():
        missing_clusters = sorted(
            assignments.loc[
                missing_review,
                "cluster",
            ].astype(str).unique()
        )
        raise RuntimeError(
            f"{cancer_type}: reviewed workbench has no annotation "
            f"for Step 16 clusters {missing_clusters}."
        )

    assignment_frames.append(assignments)

    assignment_audit_rows.append(
        {
            "cancer_type": cancer_type,
            "assignment_path": str(path),
            "n_assigned_cells": int(
                len(assignments)
            ),
            "n_clusters": int(
                assignments["cluster"].nunique()
            ),
            "cell_id_method": id_method,
            "handoff_overlap_fraction": direct_overlap,
        }
    )

tumor_assignments = pd.concat(
    assignment_frames,
    axis=0,
)

if tumor_assignments.index.duplicated().any():
    duplicates = tumor_assignments.index[
        tumor_assignments.index.duplicated(
            keep=False
        )
    ][:20].tolist()
    raise RuntimeError(
        "Cell IDs occur in more than one cancer assignment object: "
        f"{duplicates}"
    )

assignment_audit = pd.DataFrame(
    assignment_audit_rows
)

display(assignment_audit)
display(
    tumor_assignments[
        [
            "cancer_type_cluster_object",
            "cluster",
            "Specific_annotation_reviewed",
            "Broader_annotation_reviewed",
        ]
    ].head()
)

assignment_audit.to_csv(
    OUTPUT_ROOT
    / "reviewed_tumor_assignment_input_audit.csv",
    index=False,
)
tumor_assignments.to_parquet(
    OUTPUT_ROOT
    / "reviewed_tumor_cell_cluster_annotations.parquet"
)


,cancer_type,assignment_path,n_assigned_cells,n_clusters,cell_id_method,handoff_overlap_fraction
0,melanoma,/host_root/nethome/reny28/Projects/Visium_proj...,521397,19,assignment_index,1.0
1,NSCLC,/host_root/nethome/reny28/Projects/Visium_proj...,140501,10,assignment_index,1.0
2,colon_cancer,/host_root/nethome/reny28/Projects/Visium_proj...,26630,10,assignment_index,1.0


,cancer_type_cluster_object,cluster,Specific_annotation_reviewed,Broader_annotation_reviewed
cell_id,,,,
Screen_18_23::Screen_18_23_proseg_0,melanoma,16,Tumor/AP-1,Tumor
Screen_18_23::Screen_18_23_proseg_1,melanoma,14,Hepatocyte-like,Hepatocyte-like
Screen_18_23::Screen_18_23_proseg_3,melanoma,14,Hepatocyte-like,Hepatocyte-like
Screen_18_23::Screen_18_23_proseg_4,melanoma,14,Hepatocyte-like,Hepatocyte-like
Screen_18_23::Screen_18_23_proseg_6,melanoma,14,Hepatocyte-like,Hepatocyte-like


## Build the two mixed all-cell annotations

The full merged handoff is used as the authoritative cell universe.

```text
non-clustered cells
    preserve signature annotation

Step 16 clustered cells
    receive the reviewed workbench label
```

The output also retains the Leiden cluster, cancer object, baseline label, and
annotation provenance for auditing.


In [7]:
# ---------------------------------------------------------------------
# Harmonize reviewed tumor clusters with signature-based all-cell labels
# ---------------------------------------------------------------------
annotation_handoff = merged_obs[
    [
        column
        for column in [
            "source_cell_id",
            "sample",
            "patient",
            "cancer_type",
            "biopsy_stage",
            BASELINE_ANNOTATION_COLUMN,
            "qc_pass",
            "qc_total_counts",
            "qc_n_genes_by_counts",
            "qc_pct_counts_mt",
            "resolvi_diffusion_proportion",
        ]
        if column in merged_obs
    ]
].copy()

annotation_handoff.insert(
    0,
    "cell_id",
    annotation_handoff.index.astype(str),
)

annotation_handoff[
    "tumor_leiden_res_0_2"
] = ""
annotation_handoff[
    "tumor_cluster_cancer_type"
] = ""
annotation_handoff[
    "reviewed_specific_cluster_annotation"
] = ""
annotation_handoff[
    "reviewed_broader_cluster_annotation"
] = ""

baseline = (
    annotation_handoff[
        BASELINE_ANNOTATION_COLUMN
    ]
    .astype(str)
    .map(clean_annotation_value)
)

annotation_handoff[
    SPECIFIC_OUTPUT_COLUMN
] = baseline.to_numpy()
annotation_handoff[
    BROADER_OUTPUT_COLUMN
] = baseline.to_numpy()

annotation_handoff[
    "annotation_refined_by_tumor_leiden_res_0_2"
] = False
annotation_handoff[
    "annotation_provenance"
] = "signature_preserved"

refined_ids = tumor_assignments.index
annotation_handoff.loc[
    refined_ids,
    "tumor_leiden_res_0_2",
] = tumor_assignments[
    "cluster"
].astype(str).to_numpy()

annotation_handoff.loc[
    refined_ids,
    "tumor_cluster_cancer_type",
] = tumor_assignments[
    "cancer_type_cluster_object"
].astype(str).to_numpy()

annotation_handoff.loc[
    refined_ids,
    "reviewed_specific_cluster_annotation",
] = tumor_assignments[
    "Specific_annotation_reviewed"
].astype(str).to_numpy()

annotation_handoff.loc[
    refined_ids,
    "reviewed_broader_cluster_annotation",
] = tumor_assignments[
    "Broader_annotation_reviewed"
].astype(str).to_numpy()

annotation_handoff.loc[
    refined_ids,
    SPECIFIC_OUTPUT_COLUMN,
] = tumor_assignments[
    "Specific_annotation_reviewed"
].astype(str).to_numpy()

annotation_handoff.loc[
    refined_ids,
    BROADER_OUTPUT_COLUMN,
] = tumor_assignments[
    "Broader_annotation_reviewed"
].astype(str).to_numpy()

annotation_handoff.loc[
    refined_ids,
    "annotation_refined_by_tumor_leiden_res_0_2",
] = True
annotation_handoff.loc[
    refined_ids,
    "annotation_provenance",
] = "reviewed_tumor_leiden_res_0_2"

for column in [
    SPECIFIC_OUTPUT_COLUMN,
    BROADER_OUTPUT_COLUMN,
]:
    cleaned = annotation_handoff[
        column
    ].map(clean_annotation_value)

    if cleaned.eq("").any():
        examples = annotation_handoff.loc[
            cleaned.eq(""),
            [
                "cell_id",
                "sample",
                BASELINE_ANNOTATION_COLUMN,
            ],
        ].head(10)
        raise RuntimeError(
            f"{column} contains missing annotations:\n"
            + examples.to_string(index=False)
        )

    annotation_handoff[column] = cleaned

# The clustered universe is expected to originate from the broad tumor or
# unresolved labels. Report but do not silently discard any mismatch.
refined_baseline = annotation_handoff.loc[
    refined_ids,
    BASELINE_ANNOTATION_COLUMN,
].astype(str)

unexpected_refined_baseline = sorted(
    set(refined_baseline)
    .difference(
        {
            "Tumor/epithelial",
            "Other/unresolved",
        }
    )
)
if unexpected_refined_baseline:
    warnings.warn(
        "Some manually refined cells did not originate from the expected "
        "Tumor/epithelial or Other/unresolved classes: "
        f"{unexpected_refined_baseline}"
    )

# Audit cells that retain a broad tumor/unresolved signature label because they
# were not included in the QC-passing Step 16 clustering universe.
broad_candidate_mask = baseline.isin(
    {
        "Tumor/epithelial",
        "Other/unresolved",
    }
)
not_refined_mask = (
    broad_candidate_mask
    & ~annotation_handoff[
        "annotation_refined_by_tumor_leiden_res_0_2"
    ]
)

harmonization_summary = {
    "pipeline_version": PIPELINE_VERSION,
    "merged_handoff": str(
        MERGED_HANDOFF_PATH
    ),
    "n_all_cells": int(
        len(annotation_handoff)
    ),
    "n_manually_refined_cells": int(
        annotation_handoff[
            "annotation_refined_by_tumor_leiden_res_0_2"
        ].sum()
    ),
    "n_signature_preserved_cells": int(
        (
            ~annotation_handoff[
                "annotation_refined_by_tumor_leiden_res_0_2"
            ]
        ).sum()
    ),
    "n_broad_tumor_unresolved_not_refined": int(
        not_refined_mask.sum()
    ),
    "specific_output_column": (
        SPECIFIC_OUTPUT_COLUMN
    ),
    "broader_output_column": (
        BROADER_OUTPUT_COLUMN
    ),
}

print(json.dumps(
    harmonization_summary,
    indent=2,
))


{
  "pipeline_version": "2026-08-21-harmonized-signature-tumor-cluster-annotations-v1",
  "merged_handoff": "/host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/15_merged_annotation_handoff/VisiumHD_12samples_ResolVI_corrected_preliminary_annotation.zarr",
  "n_all_cells": 862427,
  "n_manually_refined_cells": 688528,
  "n_signature_preserved_cells": 173899,
  "n_broad_tumor_unresolved_not_refined": 0,
  "specific_output_column": "Specific_celltype_annotation",
  "broader_output_column": "Broader_celltype_annotation"
}


In [8]:
# ---------------------------------------------------------------------
# Harmonization audit tables
# ---------------------------------------------------------------------
specific_transition = pd.crosstab(
    annotation_handoff[
        BASELINE_ANNOTATION_COLUMN
    ].astype(str),
    annotation_handoff[
        SPECIFIC_OUTPUT_COLUMN
    ].astype(str),
)
broader_transition = pd.crosstab(
    annotation_handoff[
        BASELINE_ANNOTATION_COLUMN
    ].astype(str),
    annotation_handoff[
        BROADER_OUTPUT_COLUMN
    ].astype(str),
)

specific_transition.to_csv(
    OUTPUT_ROOT
    / "baseline_to_specific_annotation_transition.csv"
)
broader_transition.to_csv(
    OUTPUT_ROOT
    / "baseline_to_broader_annotation_transition.csv"
)

refined_cluster_counts = (
    annotation_handoff.loc[
        annotation_handoff[
            "annotation_refined_by_tumor_leiden_res_0_2"
        ]
    ]
    .groupby(
        [
            "tumor_cluster_cancer_type",
            "tumor_leiden_res_0_2",
            "reviewed_specific_cluster_annotation",
            "reviewed_broader_cluster_annotation",
        ],
        observed=True,
    )
    .size()
    .rename("n_cells")
    .reset_index()
)
refined_cluster_counts.to_csv(
    OUTPUT_ROOT
    / "reviewed_cluster_annotation_cell_counts.csv",
    index=False,
)

# Identify spelling variants that collapse to the same simplified token.
spelling_rows = []
for annotation_column in [
    SPECIFIC_OUTPUT_COLUMN,
    BROADER_OUTPUT_COLUMN,
]:
    values = sorted(
        annotation_handoff[
            annotation_column
        ].astype(str).unique()
    )
    frame = pd.DataFrame(
        {
            "annotation_column": annotation_column,
            "label": values,
        }
    )
    frame["canonical_spelling_token"] = (
        frame["label"].map(
            canonical_column_token
        )
    )
    spelling_rows.append(frame)

spelling_audit = pd.concat(
    spelling_rows,
    ignore_index=True,
)
spelling_conflicts = (
    spelling_audit.groupby(
        [
            "annotation_column",
            "canonical_spelling_token",
        ],
        observed=True,
    )["label"]
    .agg(
        n_spellings="nunique",
        spellings=lambda values: ";".join(
            sorted(set(values))
        ),
    )
    .reset_index()
)
spelling_conflicts = spelling_conflicts[
    spelling_conflicts[
        "n_spellings"
    ] > 1
]
spelling_conflicts.to_csv(
    OUTPUT_ROOT
    / "annotation_spelling_conflicts.csv",
    index=False,
)

if len(spelling_conflicts):
    warnings.warn(
        "Potential annotation spelling variants were detected. "
        "Review annotation_spelling_conflicts.csv."
    )
    display(spelling_conflicts)

display(refined_cluster_counts)


,tumor_cluster_cancer_type,tumor_leiden_res_0_2,reviewed_specific_cluster_annotation,reviewed_broader_cluster_annotation,n_cells
0,NSCLC,0,Tumor/proliferation,Tumor,13141
1,NSCLC,1,Tumor/AT-like,Tumor,1096
2,NSCLC,2,Tumor/AT-like,Tumor,57750
3,NSCLC,3,Fibroblast/stromal,Fibroblast/stromal,12340
4,NSCLC,4,Immune-mixed,Immune-mixed,18537
5,NSCLC,5,Monocyte/macrophage,Monocyte/macrophage,14591
6,NSCLC,6,Tumor/stressed,Tumor,1262
7,NSCLC,7,Tumor/proliferation,Tumor,3929
8,NSCLC,8,Tumor/stressed,Tumor,8850
9,NSCLC,9,Endothelial,Endothelial,9005


## One shared cell-type palette

The palette is created from the union of labels in both mixed annotations.
The same label receives the same color in:

```text
Specific spatial plots
Broader spatial plots
Specific stacked bars
Broader stacked bars
```

Known signature labels receive predefined colors. New reviewed labels receive
deterministic qualitative colors and are written to CSV/JSON so the palette can
be reused in later figures.


In [9]:
# ---------------------------------------------------------------------
# Build and save the shared annotation palette
# ---------------------------------------------------------------------
def qualitative_color_pool() -> list[str]:
    colors = []

    for cmap_name in [
        "tab20",
        "tab20b",
        "tab20c",
        "Set3",
        "Dark2",
        "Paired",
    ]:
        cmap = plt.get_cmap(cmap_name)
        if hasattr(cmap, "colors"):
            colors.extend(
                mcolors.to_hex(color)
                for color in cmap.colors
            )

    # Deduplicate while preserving order.
    colors = list(dict.fromkeys(colors))

    # Additional deterministic colors if the qualitative pools are exhausted.
    for index in range(120):
        hue = (
            index * 0.618033988749895
        ) % 1.0
        rgb = mcolors.hsv_to_rgb(
            (
                hue,
                0.62,
                0.88,
            )
        )
        colors.append(
            mcolors.to_hex(rgb)
        )

    return list(dict.fromkeys(colors))


all_labels = sorted(
    set(
        annotation_handoff[
            SPECIFIC_OUTPUT_COLUMN
        ].astype(str)
    )
    | set(
        annotation_handoff[
            BROADER_OUTPUT_COLUMN
        ].astype(str)
    )
)

preferred_present = [
    label
    for label in PREFERRED_CELLTYPE_ORDER
    if label in all_labels
]
remaining_labels = sorted(
    set(all_labels)
    .difference(preferred_present)
    .difference({"Other/unresolved"})
)

CELLTYPE_ORDER = [
    *preferred_present,
    *remaining_labels,
]

# Keep unresolved at the end.
if "Other/unresolved" in CELLTYPE_ORDER:
    CELLTYPE_ORDER = [
        label
        for label in CELLTYPE_ORDER
        if label != "Other/unresolved"
    ]
    CELLTYPE_ORDER.append(
        "Other/unresolved"
    )

palette = {}
palette.update(
    {
        label: color
        for label, color in KNOWN_CELLTYPE_COLORS.items()
        if label in CELLTYPE_ORDER
    }
)
palette.update(
    {
        label: color
        for label, color in CUSTOM_COLOR_OVERRIDES.items()
        if label in CELLTYPE_ORDER
    }
)
# I have a prefered color map
palette = [
            '#C12CBE', '#4D96FF', '#1F5AA6', '#8ECAE6', '#00A6A6'
            , '#F4A261', '#8C564B', '#E9C46A', '#D73027', '#F28482'
            , '#CFD266', '#D43E88', '#aec7e8', '#ff7f0e', '#ffbb78'
            , '#2ca02c', '#4F9B8F', '#7B2BBF', '#98df8a', '#000000'
        ]
palette = dict(zip(CELLTYPE_ORDER, palette))

used_colors = set(palette.values())
pool = [
    color
    for color in qualitative_color_pool()
    if color not in used_colors
]

for label in CELLTYPE_ORDER:
    if label not in palette:
        palette[label] = pool.pop(0)

palette_table = pd.DataFrame(
    {
        "cell_type": CELLTYPE_ORDER,
        "color": [
            palette[label]
            for label in CELLTYPE_ORDER
        ],
        "order": np.arange(
            len(CELLTYPE_ORDER),
            dtype=int,
        ),
    }
)

palette_table.to_csv(
    OUTPUT_ROOT
    / "celltype_color_palette.csv",
    index=False,
)
(
    OUTPUT_ROOT
    / "celltype_color_palette.json"
).write_text(
    json.dumps(
        palette,
        indent=2,
    )
)

display(palette_table)
print("Total labels in shared palette:", len(CELLTYPE_ORDER))


,cell_type,color,order
0,Treg,#C12CBE,0
1,CD4+ T,#4D96FF,1
2,CD8+ T,#1F5AA6,2
3,Other T,#8ECAE6,3
4,NK,#00A6A6,4
5,B/plasma,#F4A261,5
6,Monocyte/macrophage,#8C564B,6
7,Fibroblast/stromal,#E9C46A,7
8,Endothelial,#D73027,8
9,Keratinocyte,#F28482,9


Total labels in shared palette: 20


In [10]:
# ---------------------------------------------------------------------
# Spatial plotting helpers
# ---------------------------------------------------------------------
SPATIAL_ROOT = OUTPUT_ROOT / "spatial"
SPECIFIC_SPATIAL_ROOT = (
    SPATIAL_ROOT / "specific"
)
BROADER_SPATIAL_ROOT = (
    SPATIAL_ROOT / "broader"
)
COMPARISON_SPATIAL_ROOT = (
    SPATIAL_ROOT / "specific_vs_broader"
)

for path in [
    SPECIFIC_SPATIAL_ROOT,
    BROADER_SPATIAL_ROOT,
    COMPARISON_SPATIAL_ROOT,
]:
    path.mkdir(
        parents=True,
        exist_ok=True,
    )


def figure_size_from_coordinates(
    coordinates: np.ndarray,
) -> tuple[float, float]:
    x_range = float(
        np.nanmax(coordinates[:, 0])
        - np.nanmin(coordinates[:, 0])
    )
    y_range = float(
        np.nanmax(coordinates[:, 1])
        - np.nanmin(coordinates[:, 1])
    )

    if x_range <= 0 or y_range <= 0:
        return (
            SPATIAL_MAX_FIGURE_INCHES,
            SPATIAL_MAX_FIGURE_INCHES,
        )

    if x_range >= y_range:
        return (
            SPATIAL_MAX_FIGURE_INCHES,
            max(
                SPATIAL_MIN_FIGURE_INCHES,
                SPATIAL_MAX_FIGURE_INCHES
                * y_range
                / x_range,
            ),
        )

    return (
        max(
            SPATIAL_MIN_FIGURE_INCHES,
            SPATIAL_MAX_FIGURE_INCHES
            * x_range
            / y_range,
        ),
        SPATIAL_MAX_FIGURE_INCHES,
    )


def legend_handles_for_labels(
    labels: list[str],
) -> list[Line2D]:
    return [
        Line2D(
            [],
            [],
            marker="o",
            linestyle="",
            markersize=6,
            markerfacecolor=palette[label],
            markeredgecolor="none",
            label=label,
        )
        for label in labels
    ]


def scatter_annotation_on_axis(
    ax,
    coordinates: np.ndarray,
    labels: pd.Series,
    title: str,
):
    labels = labels.astype(str)
    present = [
        label
        for label in CELLTYPE_ORDER
        if label in set(labels)
    ]

    # Draw abundant populations first so sparse populations remain visible.
    draw_order = (
        labels.value_counts()
        .sort_values(ascending=False)
        .index.astype(str)
        .tolist()
    )
    draw_order = [
        label
        for label in draw_order
        if label in palette
    ]

    for label in draw_order:
        mask = (
            labels.to_numpy()
            == label
        )
        ax.scatter(
            coordinates[mask, 0],
            coordinates[mask, 1],
            s=SPATIAL_POINT_SIZE,
            color=palette[label],
            alpha=SPATIAL_ALPHA,
            linewidths=0,
            rasterized=True,
        )

    ax.set_title(
        title,
        fontsize=14,
    )
    ax.set_aspect(
        "equal",
        adjustable="box",
    )
    ax.invert_yaxis()
    ax.axis("off")

    return present


def save_single_spatial_plot(
    sample: str,
    annotation_column: str,
    display_name: str,
    output_path: Path,
):
    mask = (
        annotation_handoff[
            "sample"
        ].astype(str).eq(sample)
    ).to_numpy()

    coordinates = merged_spatial[mask]
    labels = annotation_handoff.loc[
        mask,
        annotation_column,
    ]

    fig, ax = plt.subplots(
        figsize=figure_size_from_coordinates(
            coordinates
        )
    )

    present = scatter_annotation_on_axis(
        ax,
        coordinates,
        labels,
        title=(
            f"{sample}\n{display_name}"
        ),
    )

    ax.legend(
        handles=legend_handles_for_labels(
            present
        ),
        loc="center left",
        bbox_to_anchor=(1.01, 0.5),
        frameon=False,
        fontsize=8,
        ncol=1,
    )

    fig.tight_layout()
    fig.savefig(
        output_path,
        dpi=PLOT_DPI,
        bbox_inches="tight",
        pad_inches=0.03,
    )
    plt.close(fig)


def save_comparison_spatial_plot(
    sample: str,
    output_path: Path,
):
    mask = (
        annotation_handoff[
            "sample"
        ].astype(str).eq(sample)
    ).to_numpy()

    coordinates = merged_spatial[mask]
    specific = annotation_handoff.loc[
        mask,
        SPECIFIC_OUTPUT_COLUMN,
    ]
    broader = annotation_handoff.loc[
        mask,
        BROADER_OUTPUT_COLUMN,
    ]

    single_width, single_height = (
        figure_size_from_coordinates(
            coordinates
        )
    )

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(
            2 * single_width,
            single_height,
        ),
    )

    specific_present = (
        scatter_annotation_on_axis(
            axes[0],
            coordinates,
            specific,
            title=(
                "Specific_celltype_annotation"
            ),
        )
    )
    broader_present = (
        scatter_annotation_on_axis(
            axes[1],
            coordinates,
            broader,
            title=(
                "Broader_celltype_annotation"
            ),
        )
    )

    present = [
        label
        for label in CELLTYPE_ORDER
        if label
        in set(
            [
                *specific_present,
                *broader_present,
            ]
        )
    ]

    fig.suptitle(
        sample,
        fontsize=16,
    )
    fig.legend(
        handles=legend_handles_for_labels(
            present
        ),
        loc="lower center",
        bbox_to_anchor=(0.5, -0.02),
        frameon=False,
        fontsize=8,
        ncol=min(
            6,
            max(1, len(present)),
        ),
    )

    fig.tight_layout(
        rect=(0, 0.08, 1, 0.96)
    )
    fig.savefig(
        output_path,
        dpi=PLOT_DPI,
        bbox_inches="tight",
        pad_inches=0.03,
    )
    plt.close(fig)


In [11]:
# ---------------------------------------------------------------------
# Generate two annotation-specific spatial maps for every sample
# ---------------------------------------------------------------------
spatial_manifest_rows = []

for sample in SAMPLE_ORDER:
    print("Spatial plots:", sample)

    specific_path = (
        SPECIFIC_SPATIAL_ROOT
        / (
            f"{sample}_"
            "Specific_celltype_annotation_spatial.png"
        )
    )
    broader_path = (
        BROADER_SPATIAL_ROOT
        / (
            f"{sample}_"
            "Broader_celltype_annotation_spatial.png"
        )
    )
    comparison_path = (
        COMPARISON_SPATIAL_ROOT
        / (
            f"{sample}_"
            "Specific_vs_Broader_spatial.png"
        )
    )

    save_single_spatial_plot(
        sample,
        SPECIFIC_OUTPUT_COLUMN,
        "Specific cell-type annotation",
        specific_path,
    )
    save_single_spatial_plot(
        sample,
        BROADER_OUTPUT_COLUMN,
        "Broader cell-type annotation",
        broader_path,
    )
    save_comparison_spatial_plot(
        sample,
        comparison_path,
    )

    spatial_manifest_rows.append(
        {
            "sample": sample,
            "specific_spatial": str(
                specific_path
            ),
            "broader_spatial": str(
                broader_path
            ),
            "comparison_spatial": str(
                comparison_path
            ),
        }
    )

spatial_manifest = pd.DataFrame(
    spatial_manifest_rows
)
spatial_manifest.to_csv(
    OUTPUT_ROOT
    / "spatial_plot_manifest.csv",
    index=False,
)

display(spatial_manifest)


Spatial plots: Screen_16_22
Spatial plots: C2D15_16_22
Spatial plots: Screen_18_23
Spatial plots: C2D15_18_23
Spatial plots: Screen_30_16
Spatial plots: C2D15_30_16
Spatial plots: Screen_17_26
Spatial plots: C2D15_17_26
Spatial plots: Screen_39_21
Spatial plots: C2D15_39_21
Spatial plots: Screen_23_25
Spatial plots: C2D15_23_25


,sample,specific_spatial,broader_spatial,comparison_spatial
0,Screen_16_22,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
1,C2D15_16_22,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
2,Screen_18_23,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
3,C2D15_18_23,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
4,Screen_30_16,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
5,C2D15_30_16,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
6,Screen_17_26,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
7,C2D15_17_26,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
8,Screen_39_21,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
9,C2D15_39_21,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...


## Composition tables and stacked bar charts

For each annotation scheme, the notebook creates 100% stacked composition
plots across four views:

```text
all 12 samples aggregated
Screen versus C2D15
patient
individual sample
```

It also writes the underlying counts and fractions. Raw-count stacked figures
are generated as an additional reference.


In [12]:
# ---------------------------------------------------------------------
# Composition tables and stacked-bar helpers
# ---------------------------------------------------------------------
COMPOSITION_ROOT = (
    OUTPUT_ROOT / "composition"
)
COMPOSITION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


def composition_matrix(
    annotation_column: str,
    grouping: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    frame = annotation_handoff[
        [
            annotation_column,
            "sample",
            "patient",
            "biopsy_stage",
        ]
    ].copy()

    if grouping == "overall":
        frame["group"] = "All 12 samples"
        group_order = ["All 12 samples"]
    elif grouping == "biopsy_stage":
        frame["group"] = (
            frame["biopsy_stage"].astype(str)
        )
        group_order = STAGE_ORDER
    elif grouping == "patient":
        frame["group"] = (
            frame["patient"].astype(str)
        )
        group_order = PATIENT_ORDER
    elif grouping == "sample":
        frame["group"] = (
            frame["sample"].astype(str)
        )
        group_order = SAMPLE_ORDER
    else:
        raise ValueError(grouping)

    counts = pd.crosstab(
        frame["group"],
        frame[annotation_column].astype(str),
    )
    counts = counts.reindex(
        index=group_order,
        columns=CELLTYPE_ORDER,
        fill_value=0,
    )

    fractions = counts.div(
        counts.sum(axis=1).replace(0, np.nan),
        axis=0,
    ).fillna(0.0)

    return counts, fractions


def save_composition_tables(
    annotation_column: str,
) -> dict[str, dict[str, pd.DataFrame]]:
    output = {}

    for grouping in [
        "overall",
        "biopsy_stage",
        "patient",
        "sample",
    ]:
        counts, fractions = (
            composition_matrix(
                annotation_column,
                grouping,
            )
        )

        safe_name = annotation_column
        counts.to_csv(
            COMPOSITION_ROOT
            / (
                f"{safe_name}_{grouping}_counts.csv"
            )
        )
        fractions.to_csv(
            COMPOSITION_ROOT
            / (
                f"{safe_name}_{grouping}_fractions.csv"
            )
        )

        output[grouping] = {
            "counts": counts,
            "fractions": fractions,
        }

    return output


def stacked_bar_on_axis(
    ax,
    matrix: pd.DataFrame,
    *,
    fraction: bool,
    title: str,
):
    bottom = np.zeros(
        len(matrix),
        dtype=float,
    )

    x = np.arange(len(matrix))

    for cell_type in CELLTYPE_ORDER:
        values = matrix[
            cell_type
        ].to_numpy(dtype=float)

        ax.bar(
            x,
            values,
            bottom=bottom,
            color=palette[cell_type],
            width=0.82,
            linewidth=0,
        )
        bottom += values

    ax.set_xticks(
        x,
        labels=matrix.index.astype(str),
        rotation=45 if len(matrix) > 2 else 0,
        ha="right" if len(matrix) > 2 else "center",
    )
    ax.set_title(title)
    ax.set_xlabel("")

    if fraction:
        ax.set_ylim(0, 1)
        ax.yaxis.set_major_formatter(
            PercentFormatter(1.0)
        )
        ax.set_ylabel("Cell fraction")
    else:
        ax.set_ylabel("Number of cells")
        ax.ticklabel_format(
            axis="y",
            style="sci",
            scilimits=(0, 0),
        )

    ax.grid(
        axis="y",
        alpha=0.20,
        linewidth=0.5,
    )


def save_four_view_stacked_figure(
    annotation_column: str,
    tables: dict,
    output_path: Path,
    *,
    fraction: bool,
):
    figure = plt.figure(
        figsize=(31, 9)
    )
    grid = GridSpec(
        1,
        4,
        figure=figure,
        width_ratios=[
            1.5,
            2.5,
            6.0,
            12.0,
        ],
        wspace=0.35,
    )

    groupings = [
        (
            "overall",
            "All cells",
        ),
        (
            "biopsy_stage",
            "Screen versus C2D15",
        ),
        (
            "patient",
            "Patient",
        ),
        (
            "sample",
            "Sample",
        ),
    ]

    for index, (
        grouping,
        title,
    ) in enumerate(groupings):
        ax = figure.add_subplot(
            grid[0, index]
        )
        matrix = tables[grouping][
            "fractions"
            if fraction
            else "counts"
        ]
        stacked_bar_on_axis(
            ax,
            matrix,
            fraction=fraction,
            title=title,
        )

    legend_handles = [
        Line2D(
            [],
            [],
            marker="s",
            linestyle="",
            markersize=9,
            markerfacecolor=palette[label],
            markeredgecolor="none",
            label=label,
        )
        for label in CELLTYPE_ORDER
    ]

    figure.legend(
        handles=legend_handles,
        loc="lower center",
        bbox_to_anchor=(0.5, -0.01),
        frameon=False,
        ncol=min(
            STACKED_BAR_LEGEND_COLUMNS,
            max(1, len(CELLTYPE_ORDER)),
        ),
        fontsize=9,
    )

    figure.suptitle(
        (
            f"{annotation_column}\n"
            + (
                "100% stacked cell-type composition"
                if fraction
                else "stacked cell counts"
            )
        ),
        fontsize=17,
    )

    figure.subplots_adjust(
        bottom=0.26,
        top=0.84,
        left=0.04,
        right=0.99,
    )
    figure.savefig(
        output_path,
        dpi=PLOT_DPI,
        bbox_inches="tight",
    )
    plt.close(figure)


In [13]:
# ---------------------------------------------------------------------
# Generate Specific and Broader composition outputs
# ---------------------------------------------------------------------
composition_outputs = {}

for annotation_column in [
    SPECIFIC_OUTPUT_COLUMN,
    BROADER_OUTPUT_COLUMN,
]:
    tables = save_composition_tables(
        annotation_column
    )
    composition_outputs[
        annotation_column
    ] = tables

    if MAKE_FRACTION_STACKED_BARS:
        save_four_view_stacked_figure(
            annotation_column,
            tables,
            COMPOSITION_ROOT
            / (
                f"{annotation_column}_"
                "overall_stage_patient_sample_100pct_stacked.png"
            ),
            fraction=True,
        )

    if MAKE_COUNT_STACKED_BARS:
        save_four_view_stacked_figure(
            annotation_column,
            tables,
            COMPOSITION_ROOT
            / (
                f"{annotation_column}_"
                "overall_stage_patient_sample_count_stacked.png"
            ),
            fraction=False,
        )

print("Composition outputs written to:", COMPOSITION_ROOT)

# Concise preview of sample-level fractions.
display(
    composition_outputs[
        SPECIFIC_OUTPUT_COLUMN
    ]["sample"]["fractions"]
)
display(
    composition_outputs[
        BROADER_OUTPUT_COLUMN
    ]["sample"]["fractions"]
)


Composition outputs written to: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/18_harmonized_signature_and_tumor_cluster_annotations/composition


Specific_celltype_annotation,Treg,CD4+ T,CD8+ T,Other T,NK,B/plasma,Monocyte/macrophage,Fibroblast/stromal,Endothelial,Keratinocyte,Tumor,Hepatocyte-like,Immune-mixed,Tumor/AP-1,Tumor/AT-like,Tumor/differentiated,Tumor/hypoxia,Tumor/progenitor-like,Tumor/proliferation,Tumor/stressed
group,,,,,,,,,,,,,,,,,,,,
Screen_16_22,0.000028,0.001789,0.001718,0.014126,0.000000,0.085309,0.072873,0.331284,0.070118,0.074037,0.0,0.000028,0.000000,0.000043,0.000000,0.281411,0.000000,0.000000,0.067222,0.000014
C2D15_16_22,0.001670,0.004580,0.004293,0.022784,0.000326,0.073180,0.077264,0.279945,0.052993,0.032988,0.0,0.000026,0.000000,0.000078,0.000000,0.340845,0.000000,0.000000,0.108974,0.000052
Screen_18_23,0.000000,0.002307,0.001989,0.008455,0.001340,0.052994,0.085541,0.005360,0.187149,0.000000,0.0,0.466458,0.000000,0.186928,0.000000,0.001160,0.000000,0.000000,0.000276,0.000041
C2D15_18_23,0.000323,0.010272,0.010595,0.022964,0.005002,0.036410,0.071475,0.053996,0.243304,0.000000,0.0,0.409541,0.000000,0.133323,0.000000,0.001613,0.000000,0.000000,0.001129,0.000054
Screen_30_16,0.003061,0.008152,0.008794,0.041925,0.036341,0.027577,0.000284,0.093077,0.330576,0.000030,0.0,0.000015,0.000000,0.000015,0.000000,0.138764,0.000000,0.000000,0.083700,0.227690
C2D15_30_16,0.000057,0.000244,0.000165,0.000776,0.000000,0.001350,0.113102,0.081837,0.072473,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.550192,0.000000,0.000000,0.118903,0.060901
Screen_17_26,0.000647,0.002025,0.001837,0.011233,0.000021,0.001942,0.047415,0.183648,0.081760,0.000000,0.0,0.000000,0.117358,0.000000,0.414607,0.000000,0.000000,0.000000,0.068231,0.069275
C2D15_17_26,0.000489,0.001160,0.001160,0.007325,0.000000,0.000512,0.023614,0.188778,0.054622,0.000000,0.0,0.000000,0.102636,0.000000,0.430801,0.000000,0.000000,0.000000,0.112270,0.076633
Screen_39_21,0.004009,0.014161,0.013420,0.057255,0.067495,0.087756,0.475904,0.001786,0.073115,0.000000,0.0,0.000000,0.031678,0.000000,0.000566,0.000000,0.000000,0.000000,0.171242,0.001612


Broader_celltype_annotation,Treg,CD4+ T,CD8+ T,Other T,NK,B/plasma,Monocyte/macrophage,Fibroblast/stromal,Endothelial,Keratinocyte,Tumor,Hepatocyte-like,Immune-mixed,Tumor/AP-1,Tumor/AT-like,Tumor/differentiated,Tumor/hypoxia,Tumor/progenitor-like,Tumor/proliferation,Tumor/stressed
group,,,,,,,,,,,,,,,,,,,,
Screen_16_22,0.000028,0.001789,0.001718,0.014126,0.000000,0.085309,0.072873,0.331284,0.070118,0.074037,0.348690,0.000028,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
C2D15_16_22,0.001670,0.004580,0.004293,0.022784,0.000326,0.073180,0.077264,0.279945,0.052993,0.032988,0.449950,0.000026,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Screen_18_23,0.000000,0.002307,0.001989,0.008455,0.001340,0.052994,0.085541,0.005360,0.187149,0.000000,0.188407,0.466458,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
C2D15_18_23,0.000323,0.010272,0.010595,0.022964,0.005002,0.036410,0.071475,0.053996,0.243304,0.000000,0.136119,0.409541,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Screen_30_16,0.003061,0.008152,0.008794,0.041925,0.036341,0.027577,0.000284,0.093077,0.330576,0.000030,0.450169,0.000015,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
C2D15_30_16,0.000057,0.000244,0.000165,0.000776,0.000000,0.001350,0.113102,0.081837,0.072473,0.000000,0.729996,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Screen_17_26,0.000647,0.002025,0.001837,0.011233,0.000021,0.001942,0.047415,0.183648,0.081760,0.000000,0.552113,0.000000,0.117358,0.0,0.0,0.0,0.0,0.0,0.0,0.0
C2D15_17_26,0.000489,0.001160,0.001160,0.007325,0.000000,0.000512,0.023614,0.188778,0.054622,0.000000,0.619704,0.000000,0.102636,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Screen_39_21,0.004009,0.014161,0.013420,0.057255,0.067495,0.087756,0.475904,0.001786,0.073115,0.000000,0.173420,0.000000,0.031678,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [14]:
# ---------------------------------------------------------------------
# Save all-cell annotation handoff tables
# ---------------------------------------------------------------------
DETAILED_HANDOFF_PARQUET = (
    OUTPUT_ROOT
    / "all_cells_mixed_annotation_handoff.parquet"
)
DETAILED_HANDOFF_CSV = (
    OUTPUT_ROOT
    / "all_cells_mixed_annotation_handoff.csv.gz"
)
MINIMAL_UPDATE_PARQUET = (
    OUTPUT_ROOT
    / "all_cells_annotation_update_minimal.parquet"
)
MINIMAL_UPDATE_CSV = (
    OUTPUT_ROOT
    / "all_cells_annotation_update_minimal.csv.gz"
)
SUMMARY_JSON = (
    OUTPUT_ROOT
    / "harmonized_annotation_summary.json"
)

if not OVERWRITE:
    existing = [
        path
        for path in [
            DETAILED_HANDOFF_PARQUET,
            DETAILED_HANDOFF_CSV,
            MINIMAL_UPDATE_PARQUET,
            MINIMAL_UPDATE_CSV,
            SUMMARY_JSON,
        ]
        if path.exists()
    ]
    if existing:
        raise FileExistsError(
            "Output files already exist. Set OVERWRITE=True to replace:\n"
            + "\n".join(str(path) for path in existing)
        )

annotation_handoff.to_parquet(
    DETAILED_HANDOFF_PARQUET,
    index=False,
)
annotation_handoff.to_csv(
    DETAILED_HANDOFF_CSV,
    index=False,
    compression="gzip",
)

minimal_update = annotation_handoff[
    [
        "cell_id",
        SPECIFIC_OUTPUT_COLUMN,
        BROADER_OUTPUT_COLUMN,
    ]
].copy()
minimal_update = minimal_update.set_index(
    "cell_id",
    drop=True,
)
minimal_update.index.name = "cell_id"

minimal_update.to_parquet(
    MINIMAL_UPDATE_PARQUET,
)
minimal_update.to_csv(
    MINIMAL_UPDATE_CSV,
    compression="gzip",
)

# Read-back validation.
detailed_check = pd.read_parquet(
    DETAILED_HANDOFF_PARQUET
)
minimal_check = pd.read_parquet(
    MINIMAL_UPDATE_PARQUET
)

if len(detailed_check) != len(annotation_handoff):
    raise RuntimeError(
        "Detailed handoff row count changed during write/read."
    )
if len(minimal_check) != len(annotation_handoff):
    raise RuntimeError(
        "Minimal update row count changed during write/read."
    )
if not minimal_check.index.is_unique:
    raise RuntimeError(
        "Minimal update cell IDs are not unique."
    )

harmonization_summary.update(
    {
        "n_specific_labels": int(
            annotation_handoff[
                SPECIFIC_OUTPUT_COLUMN
            ].nunique()
        ),
        "n_broader_labels": int(
            annotation_handoff[
                BROADER_OUTPUT_COLUMN
            ].nunique()
        ),
        "detailed_handoff_parquet": str(
            DETAILED_HANDOFF_PARQUET
        ),
        "minimal_update_parquet": str(
            MINIMAL_UPDATE_PARQUET
        ),
        "palette_csv": str(
            OUTPUT_ROOT
            / "celltype_color_palette.csv"
        ),
        "spatial_plot_manifest": str(
            OUTPUT_ROOT
            / "spatial_plot_manifest.csv"
        ),
    }
)

SUMMARY_JSON.write_text(
    json.dumps(
        harmonization_summary,
        indent=2,
    )
)

print("Wrote:", DETAILED_HANDOFF_PARQUET)
print("Wrote:", DETAILED_HANDOFF_CSV)
print("Wrote:", MINIMAL_UPDATE_PARQUET)
print("Wrote:", MINIMAL_UPDATE_CSV)
print("Wrote:", SUMMARY_JSON)


Wrote: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/18_harmonized_signature_and_tumor_cluster_annotations/all_cells_mixed_annotation_handoff.parquet
Wrote: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/18_harmonized_signature_and_tumor_cluster_annotations/all_cells_mixed_annotation_handoff.csv.gz
Wrote: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/18_harmonized_signature_and_tumor_cluster_annotations/all_cells_annotation_update_minimal.parquet
Wrote: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/18_harmonized_signature_and_tumor_cluster_annotations/all_cells_annotation_update_minimal.csv.gz
Wrote: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/18_harmonized_signature_and_tumor_cluster_annotations/harmonized_annotat

# Updating the merged all-sample object later

The minimal sidecar is indexed by the merged cell ID:

```python
updates = pd.read_parquet(
    ".../all_cells_annotation_update_minimal.parquet"
)
```

To attach the columns to an in-memory AnnData object:

```python
for column in [
    "Specific_celltype_annotation",
    "Broader_celltype_annotation",
]:
    if column in adata.obs:
        adata.obs = adata.obs.drop(columns=[column])

adata.obs = adata.obs.join(
    updates[
        [
            "Specific_celltype_annotation",
            "Broader_celltype_annotation",
        ]
    ],
    how="left",
    validate="one_to_one",
)

assert adata.obs[
    [
        "Specific_celltype_annotation",
        "Broader_celltype_annotation",
    ]
].notna().all().all()
```

For the large corrected-expression handoff Zarr, the safest approach is to join
this sidecar during a new export/write pass rather than materializing the full
dense object solely to edit `.obs`.

## Annotation provenance

```text
signature_preserved
    the original broad signature annotation was retained

reviewed_tumor_leiden_res_0_2
    the cell was part of the Step 16 tumor/unresolved clustering and received
    the reviewed cluster annotation from the edited Step 17 workbench
```

The detailed handoff retains both the baseline annotation and all relevant
cluster-review provenance.


In [15]:
# ---------------------------------------------------------------------
# Final output inventory
# ---------------------------------------------------------------------
for path in sorted(
    OUTPUT_ROOT.rglob("*")
):
    if path.is_file():
        print(
            f"{path.relative_to(OUTPUT_ROOT)!s:110s} "
            f"{path.stat().st_size / 1024**2:9.2f} MiB"
        )


all_cells_annotation_update_minimal.csv.gz                                                                          2.57 MiB
all_cells_annotation_update_minimal.parquet                                                                         4.87 MiB
all_cells_mixed_annotation_handoff.csv.gz                                                                          16.06 MiB
all_cells_mixed_annotation_handoff.parquet                                                                         17.50 MiB
annotation_spelling_conflicts.csv                                                                                   0.00 MiB
baseline_to_broader_annotation_transition.csv                                                                       0.00 MiB
baseline_to_specific_annotation_transition.csv                                                                      0.00 MiB
celltype_color_palette.csv                                                                                          0.00 MiB
